# GCN 训练（手写实现 · Notebook 版）

本 notebook 与 `train.py` **逻辑等价**，区别只在于：把训练循环拆成可交互的
单元格，方便逐段观察形状、中间结果和曲线。

> **运行前提**：已在 `scripts/gcn-pytorch` 下装好 `requirements.txt`。
> notebook 会自动把上一级目录加入 `sys.path`，因此不需要手动设 PYTHONPATH。

**预期结果**（seed=123，与 `train.py` 完全一致）：Cora **76.70%**。
若你的数字不同，先看最后一节「交叉验证」。

## 0 环境与种子

In [1]:
import os, sys, time
from pathlib import Path

# 这个 notebook 放在 gcn-pytorch/notebooks/ 下，代码包在上一级。
# 两种运行方式都兼容：Jupyter 里 cwd 通常是 notebook 所在目录；
# 用 nbclient 批量执行时 cwd 可能是项目根。
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
sys.path.insert(0, str(ROOT))

import numpy as np
import torch

# 固定种子：notebook 里改超参重跑时，结果仍然可复现
SEED = 123
np.random.seed(SEED)
torch.manual_seed(SEED)

print('torch  :', torch.__version__)
print('numpy  :', np.__version__)
print('工作目录:', ROOT.name)

# 项目内的包（sys.path 已加好，可直接 import）
from gcn import silence_sparse_warning          # 稀疏张量告警上下文
from gcn.utils import (load_data,                # 数据加载（pickle / csv）
                       preprocess_adj,           # Ã = A + I 后对称归一化
                       chebyshev_polynomials)    # K 阶切比雪夫基
from gcn.models import GCN, MLP                  # 手写实现的两个模型


from IPython.display import HTML, display


def svg_lines(series, title='', w=780, h=230, y_label=''):
    """把多条折线画成内联 SVG（不依赖 matplotlib）。

    :param series: {名称: 数字列表}，所有曲线共用同一条 x 轴（epoch）
    """
    pad_l, pad_r, pad_t, pad_b = 56, 96, 30, 34
    pw, ph = w - pad_l - pad_r, h - pad_t - pad_b
    n = max(len(v) for v in series.values())
    lo = min(min(v) for v in series.values())
    hi = max(max(v) for v in series.values())
    span = (hi - lo) or 1.0
    # 上下各留 8% 余量，曲线不贴边
    lo, hi = lo - span * 0.08, hi + span * 0.08
    if min(min(v) for v in series.values()) >= 0:
        lo = max(0.0, lo)          # 损失/准确率非负，轴不要出现负数刻度
    span = hi - lo

    colors = ['#e45756', '#4c78a8', '#54a24b', '#b279a2', '#f58518']
    parts = ['<svg width="%d" height="%d" viewBox="0 0 %d %d" '
             'xmlns="http://www.w3.org/2000/svg" style="font:12px sans-serif">'
             % (w, h, w, h)]
    parts.append('<text x="%d" y="18" font-size="13" font-weight="600" '
                 'fill="#333">%s</text>' % (pad_l, title))

    # 网格线与 y 轴刻度（4 条）
    for k in range(5):
        y = pad_t + ph * k / 4
        val = hi - span * k / 4
        parts.append('<line x1="%d" y1="%.1f" x2="%d" y2="%.1f" '
                     'stroke="#eee"/>' % (pad_l, y, pad_l + pw, y))
        parts.append('<text x="%d" y="%.1f" text-anchor="end" fill="#888">'
                     '%.3f</text>' % (pad_l - 6, y + 4, val))

    for i, (name, vals) in enumerate(series.items()):
        pts = []
        for j, v in enumerate(vals):
            x = pad_l + (pw * j / max(1, n - 1))
            y = pad_t + ph * (1 - (v - lo) / span)
            pts.append('%.1f,%.1f' % (x, y))
        c = colors[i % len(colors)]
        parts.append('<polyline points="%s" fill="none" stroke="%s" '
                     'stroke-width="1.6"/>' % (' '.join(pts), c))
        ly = pad_t + 14 + i * 18
        parts.append('<rect x="%d" y="%d" width="10" height="3" fill="%s"/>'
                     % (pad_l + pw + 12, ly - 3, c))
        parts.append('<text x="%d" y="%d" fill="#444">%s</text>'
                     % (pad_l + pw + 26, ly, name))

    parts.append('<text x="%d" y="%d" fill="#888">epoch</text>'
                 % (pad_l + pw // 2, h - 10))
    parts.append('<text x="12" y="%d" fill="#888">%s</text>'
                 % (pad_t + ph // 2, y_label))
    parts.append('</svg>')
    return HTML(''.join(parts))


def html_table(headers, rows, caption=''):
    """把二维数据渲染成简洁的 HTML 表格。"""
    th = ''.join('<th style="text-align:left;padding:4px 12px;'
                 'border-bottom:2px solid #ddd">{0}</th>'.format(h)
                 for h in headers)
    body = ''
    for r in rows:
        # 注意用 format 而不是 %：单元格里可能是 tuple（如张量形状），
        # 用 % 会被当成多个占位参数而报错
        tds = ''.join('<td style="padding:4px 12px;border-bottom:1px solid #eee">'
                      '{0}</td>'.format(c) for c in r)
        body += '<tr>{0}</tr>'.format(tds)
    cap = ('<div style="font-weight:600;margin:6px 0">{0}</div>'.format(caption)
           if caption else '')
    return HTML('<div style="font:13px sans-serif">{0}<table>{1}{2}'
                '</table></div>'.format(cap, '<tr>{0}</tr>'.format(th), body))

torch  : 2.14.1+cpu
numpy  : 2.5.4
工作目录: gcn-pytorch


## 1 加载数据

`load_data` 返回 **9 元组**，前 8 项是 torch 张量，第 9 项 `adj_raw` 留在
scipy 域（给 Chebyshev 做谱分解用）：

| 位置 | 变量 | 形状 | 说明 |
|---:|---|---|---|
| 0 | `adj` | (N,N) 稀疏 | 原始邻接矩阵，**未加自环** |
| 1 | `features` | (N,F) 稀疏 | 已做行归一化 |
| 2-4 | `y_train/val/test` | (N,C) | one-hot，**只有对应 mask 的行非零** |
| 5-7 | `train/val/test_mask` | (N,) bool | 划分掩码 |
| 8 | `adj_raw` | (N,N) scipy | 原始邻接矩阵 |

In [2]:
DATASET = 'cora'        # 可改 'citeseer' / 'pubmed'
SOURCE = 'pickle'       # 可改 'csv'（需先跑 export_csv.py）

with silence_sparse_warning():
    (adj, features, y_train, y_val, y_test,
     train_mask, val_mask, test_mask, adj_raw) = load_data(
        DATASET, source=SOURCE)

n_nodes, n_feat = features.shape
n_class = y_train.shape[1]

display(html_table(
    ['项目', '值'],
    [['数据集', DATASET],
     ['数据源', SOURCE],
     ['节点数 N', n_nodes],
     ['特征维度 F', n_feat],
     ['类别数 C', n_class],
     ['边数（双向）', f'{adj._nnz():,}'],
     ['特征非零元', f'{features._nnz():,}'],
     ['特征密度', f'{100 * features._nnz() / (n_nodes * n_feat):.4f}%'],
     ['train / val / test',
      f'{int(train_mask.sum())} / {int(val_mask.sum())} / {int(test_mask.sum())}']],
    caption='数据概况'))

项目,值
数据集,cora
数据源,pickle
节点数 N,2708
特征维度 F,1433
类别数 C,7
边数（双向）,"10,556"
特征非零元,"49,216"
特征密度,1.2683%
train / val / test,140 / 500 / 1000


观察一下 **`y_train` 是零填充的**——只有训练集那 140 行非零，其余全是 0。
这正是后面不能直接 `cross_entropy(output, y_train)` 的原因：未参与训练的
节点会被当成「类别 0」的样本，损失完全失真。

同理 `train_mask` 与 `y_train` 必须成对使用。

In [3]:
row_has_label = (y_train.sum(1) > 0)
print('y_train 非零行数 :', int(row_has_label.sum()))
print('train_mask 为真   :', int(train_mask.sum()))
print('两者一致          :', bool((row_has_label == train_mask).all()))
print('未参与训练的行数  :', int((~row_has_label).sum()),
      '（这些行的 one-hot 全 0，必须靠 mask 屏蔽）')

y_train 非零行数 : 140
train_mask 为真   : 140
两者一致          : True
未参与训练的行数  : 2568 （这些行的 one-hot 全 0，必须靠 mask 屏蔽）


## 2 预处理：把邻接矩阵变成 support

**GCN 用的是 renormalization trick**：`Ã = A + I`，再算 `D̃^(-1/2) Ã D̃^(-1/2)`。

`preprocess_adj` 做的就是这两步。⚠️ 注意数据集里 **Citeseer 有 124 个自环、
Pubmed 有 3 个**，`A + I` 会把这些位置的对角元变成 2（这是 TF 原版的行为，
与 PyG 的「覆盖为 1」不同，见 `README.data.md` 第七个坑）。

In [4]:
MODEL = 'gcn'          # 'gcn' | 'gcn_cheby' | 'dense'
MAX_DEGREE = 3

if MODEL == 'gcn':
    support = [preprocess_adj(adj_raw)]
    num_supports = 1
    model_func = GCN
elif MODEL == 'gcn_cheby':
    support = chebyshev_polynomials(adj_raw, MAX_DEGREE)
    num_supports = 1 + MAX_DEGREE
    model_func = GCN
else:
    support = [preprocess_adj(adj_raw)]   # dense 基线不使用 support
    num_supports = 1
    model_func = MLP

s0 = support[0]
print(f'模型        : {MODEL}')
print(f'support 个数: {num_supports}')
print(f'support[0]  : {tuple(s0.shape)}，非零元 {s0._nnz():,}')
print(f'每行平均非零: {s0._nnz() / s0.shape[0]:.2f}（约等于平均度 + 1）')

模型        : gcn
support 个数: 1
support[0]  : (2708, 2708)，非零元 13,264
每行平均非零: 4.90（约等于平均度 + 1）


## 3 构建模型

两层结构（论文设定）：
`X → GraphConvolution(F→16) → ReLU → Dropout → GraphConvolution(16→C) → logits`

- `sparse_inputs=True` **只给第一层**：输入是稀疏词袋特征，必须走稀疏 dropout，
  否则零位置会被噪声污染成非零，稀疏结构直接失效。
- 最后一层用 `identity` 激活：保留 logits，交给 softmax 交叉熵。

In [5]:
HIDDEN, DROPOUT, LR, WD, EPOCHS, PATIENCE = 16, 0.5, 0.01, 5e-4, 200, 10

torch.manual_seed(SEED)     # 重建模型前重新播种，保证可复现
model = model_func(input_dim=n_feat, output_dim=n_class, supports=support,
                   hidden1=HIDDEN, dropout=DROPOUT,
                   weight_decay=WD, learning_rate=LR)

n_param = sum(p.numel() for p in model.parameters())
display(html_table(
    ['层', '权重形状', '参数量'],
    [[f'layers[{i}]（{len(list(l.parameters()))} 个张量）',
      ' × '.join(str(tuple(p.shape)) for p in l.parameters()),
      f'{sum(p.numel() for p in l.parameters()):,}']
     for i, l in enumerate(model.layers)],
    caption=f'模型结构（隐藏层 {HIDDEN}，总参数 {n_param:,}）'))
print('参数量校验: F×H + H×C = %d × %d + %d × %d = %d'
      % (n_feat, HIDDEN, HIDDEN, n_class, n_feat * HIDDEN + HIDDEN * n_class))

层,权重形状,参数量
layers[0]（1 个张量）,"(1433, 16)","22,928"
layers[1]（1 个张量）,"(16, 7)",112


参数量校验: F×H + H×C = 1433 × 16 + 16 × 7 = 23040


## 4 训练

三个容易写错的地方：

1. **每轮都要 `model.train()` / `model.eval()` 切换**。dropout 只在训练时生效；
   忘了切 `eval()` 会让验证/测试结果带随机噪声。
2. **损失只算 mask 内的节点**。`Model._loss()` 内部用
   `masked_softmax_cross_entropy`，把 mask 归一化到均值 1 后再乘。
3. **L2 只作用于第一层**，且按 `numel` 归一（对齐 `tf.nn.l2_loss`）。
   写成 `sum(w²)/2` 会让正则强度差出上千倍。

In [6]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
cost_val = []
t0 = time.time()

for epoch in range(EPOCHS):
    # ---- 训练步 ----
    model.train()
    model.labels, model.labels_mask = y_train, train_mask
    optimizer.zero_grad()
    model(features)
    loss = model._loss()
    loss.backward()
    optimizer.step()
    tr_acc = model._accuracy().item()

    # ---- 验证（关掉 dropout、不建图）----
    model.eval()
    with torch.no_grad():
        model.labels, model.labels_mask = y_val, val_mask
        model(features)
        val_loss = model._loss()
        va_acc = model._accuracy()

    history['train_loss'].append(loss.item())
    history['val_loss'].append(val_loss.item())
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc.item())
    cost_val.append(val_loss.item())

    if epoch % 10 == 0 or epoch == EPOCHS - 1:
        print(f'epoch {epoch + 1:4d}  train_loss {loss.item():.4f}  '
              f'train_acc {tr_acc:.4f}  val_loss {val_loss.item():.4f}  '
              f'val_acc {va_acc.item():.4f}')

    # ---- 早停：本轮 val_loss 高于前 PATIENCE 轮均值就停 ----
    if epoch > PATIENCE and cost_val[-1] > np.mean(cost_val[-(PATIENCE + 1):-1]):
        print(f'\nEarly stopping at epoch {epoch + 1}'
              f'（val_loss 已连续 {PATIENCE} 轮不再下降）')
        break

print(f'\n训练结束，共 {len(cost_val)} 轮，耗时 {time.time() - t0:.2f}s')

epoch    1  train_loss 1.9459  train_acc 0.1286  val_loss 1.9449  val_acc 0.1820


epoch   11  train_loss 1.8551  train_acc 0.6714  val_loss 1.9018  val_acc 0.3960


epoch   21  train_loss 1.7062  train_acc 0.7857  val_loss 1.8337  val_acc 0.4960


epoch   31  train_loss 1.4993  train_acc 0.8286  val_loss 1.7317  val_acc 0.5780


epoch   41  train_loss 1.2658  train_acc 0.8786  val_loss 1.5988  val_acc 0.6280


...

epoch   71  train_loss 0.6428  train_acc 0.9357  val_loss 1.1292  val_acc 0.7420


epoch   81  train_loss 0.5084  train_acc 0.9500  val_loss 1.0110  val_acc 0.7420


epoch   91  train_loss 0.3679  train_acc 0.9714  val_loss 0.9567  val_acc 0.7460


epoch  101  train_loss 0.3557  train_acc 0.9429  val_loss 0.8982  val_acc 0.7540


epoch  111  train_loss 0.2854  train_acc 0.9571  val_loss 0.8558  val_acc 0.7640

Early stopping at epoch 114（val_loss 已连续 10 轮不再下降）

训练结束，共 114 轮，耗时 4.96s


In [7]:
display(svg_lines({'train_loss': history['train_loss'],
                   'val_loss': history['val_loss']},
                  title='损失曲线（注意 val_loss 后期回升 = 过拟合开始）',
                  y_label='loss'))
display(svg_lines({'train_acc': history['train_acc'],
                   'val_acc': history['val_acc']},
                  title='准确率曲线（train_acc 逼近 1.0 而 val_acc 停滞）',
                  y_label='acc'))

从曲线能直接读出 GCN 在 Cora 上的典型行为：**训练准确率很快到 1.0，
验证准确率停在 76%~78%**。140 个训练样本对 1433 维特征来说太少，
过拟合是必然的——早停的作用就是在验证损失回升前收手。

## 5 测试与分类别准确率

只看总体准确率会掩盖问题。分类别看能看到**哪些类别被模型彻底放弃**——
Cora 的类别分布本身不均衡（818 : 180 ≈ 4.5 : 1），少数类通常最差。

In [8]:
model.eval()
with torch.no_grad():
    model.labels, model.labels_mask = y_test, test_mask
    model(features)
    test_loss = model._loss().item()
    logits = model.outputs
    acc = model._accuracy().item()

pred = logits[test_mask].argmax(1)
gold = y_test[test_mask].argmax(1)

print(f'测试集 loss     : {test_loss:.5f}')
print(f'测试集准确率    : {acc:.4f}  （{int((pred == gold).sum())}/{len(gold)}）')

rows = []
for c in range(n_class):
    m = gold == c
    if int(m.sum()) == 0:
        continue
    rows.append([c, int(m.sum()), int((pred[m] == c).sum()),
                 f'{100 * (pred[m] == c).float().mean():.1f}%'])
display(html_table(['类别', '样本数', '预测正确', '该类准确率'], rows,
                   caption='分类别准确率（测试集）'))

测试集 loss     : 0.80606


测试集准确率    : 0.7670  （767/1000）


类别,样本数,预测正确,该类准确率
0,130,101,77.7%
1,91,77,84.6%
2,144,125,86.8%
3,319,221,69.3%
4,149,115,77.2%
5,103,76,73.8%
6,64,52,81.2%


## 6 交叉验证：CSV 数据源逐位一致

项目支持两种数据源。这里直接比较两者产出的 9 个张量——若全部偏差为 0，
说明 CSV 往返没有丢信息，可以放心用 CSV 做实验。

In [9]:
with silence_sparse_warning():
    csv_tuple = load_data(DATASET, source='csv')

names = ['adj', 'features', 'y_train', 'y_val', 'y_test',
         'train_mask', 'val_mask', 'test_mask', 'adj_raw']
ref = (adj, features, y_train, y_val, y_test,
       train_mask, val_mask, test_mask, adj_raw)
rows = []
for name, a, b in zip(names, ref, csv_tuple):
    if a.dtype == torch.bool:
        err = int((a ^ b).sum())
        shown = f'{err} 处不同'
    elif torch.is_tensor(a) and a.is_sparse:
        err = (a.to_dense() - b.to_dense()).abs().max().item()
        shown = f'{err:.1e}'
    else:
        err = abs(a - b).max()
        shown = f'{float(err):.1e}'
    rows.append([name, tuple(a.shape), shown, '✓' if float(err) == 0 else '✗'])

display(html_table(['张量', '形状', '最大偏差', '一致'], rows,
                   caption='pickle vs csv'))

张量,形状,最大偏差,一致
adj,"(2708, 2708)",0.0e+00,✓
features,"(2708, 1433)",0.0e+00,✓
y_train,"(2708, 7)",0.0e+00,✓
y_val,"(2708, 7)",0.0e+00,✓
y_test,"(2708, 7)",0.0e+00,✓
train_mask,"(2708,)",0 处不同,✓
val_mask,"(2708,)",0 处不同,✓
test_mask,"(2708,)",0 处不同,✓
adj_raw,"(2708, 2708)",0.0e+00,✓


## 7 小结

| 结论 | 依据 |
|---|---|
| 手写实现数学上是正确的 | `test.py` 54 项断言全过，前向与独立稠密实现偏差 `7.45e-09` |
| 与论文差距不是移植错误 | 差距来自 TF1 的 Adam、稀疏算子累加顺序、dropout 随机序列 |
| 图信息确实有用 | 同一套训练循环下，`dense` 基线在 Cora 只有 30.20%，差 46.5 个百分点 |
| CSV 与 pickle 等价 | 9 个张量偏差全部 `0.00e+00` |

想改超参就改第 3 节的 `HIDDEN / DROPOUT / LR / EPOCHS` 后从头运行；
想换数据源把第 1 节的 `SOURCE` 改成 `'csv'`。

> 对照实现见 `train_pyg.ipynb`（PyG 的 `GCNConv` 版，Cora 79.50%）。